In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
# @title 1.1 Choix du Backend Géométrique
# 'geogram' est recommandé pour la vitesse (headless). 'cgal' est plus lent mais exact.
BACKEND = 'geogram'  # @param ['geogram', 'cgal']
print(f"Backend sélectionné : {BACKEND}")

In [ ]:
# @title 1.2 Installation des dépendances système
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git libeigen3-dev libomp-dev libtbb-dev libtbbmalloc2

if BACKEND == 'cgal':
    # libboost-all-dev est souvent nécessaire pour que CMake détecte correctement CGAL
    !apt-get install -y -qq libcgal-dev libgmp-dev libmpfr-dev libboost-all-dev

In [ ]:
# @title 1.3 Installation des dépendances Python
!pip install -q --upgrade pip setuptools wheel Cython cmake jedi gdown pybind11
!pip install -q numpy scipy scikit-learn matplotlib umap-learn


In [ ]:
%%bash
# @title 1.4 Clone HGP-clusterer
set -euo pipefail
WORKDIR="/content"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

if [ -d HGP-clusterer ]; then
    git -C HGP-clusterer pull --ff-only
else
    git clone https://github.com/Ludwig-H/HGP-clusterer.git
fi

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
# @title 1.5 Compilation de HGP-Reducer
import os
import sys
import subprocess

WORKDIR = "/content"
os.chdir(WORKDIR)

if BACKEND == 'geogram':
    if not os.path.exists('geogram'):
        print("Clonage de Geogram...")
        !git clone --recursive https://github.com/BrunoLevy/geogram.git

    print("Compilation de Geogram (Headless)...")
    !rm -rf geogram/build
    !cmake -S geogram -B geogram/build -DCMAKE_BUILD_TYPE=Release -DGEOGRAM_WITH_GRAPHICS=OFF -DGEOGRAM_WITH_LUA=OFF -DGEOGRAM_WITH_GARGANTUA=OFF
    !cmake --build geogram/build --config Release --parallel 4
    !cmake --install geogram/build --prefix /usr/local
    !ldconfig
    os.environ['GEOGRAM_INSTALL_PREFIX'] = '/usr/local'

elif BACKEND == 'cgal':
    print("Configuration CGAL...")

    # -- FIX: Add CGAL path to environment for setup_cgal.py --
    cgal_prefix = "/usr/lib/x86_64-linux-gnu/cmake/CGAL"
    current_cpp = os.environ.get("CMAKE_PREFIX_PATH", "")
    os.environ["CMAKE_PREFIX_PATH"] = f"{current_cpp}:{cgal_prefix}" if current_cpp else cgal_prefix
    # ---------------------------------------------------------

    try:
        subprocess.run(["python3", f"{WORKDIR}/HGP-clusterer/scripts/setup_cgal.py"], check=True)
    except subprocess.CalledProcessError:
        print("\u26a0\ufe0f Attention: Echec du script setup_cgal.py. Tentative de continuation avec le build principal...")

os.chdir(f"{WORKDIR}/HGP-clusterer")
!rm -rf build dist *.egg-info

install_cmd = "pip install --no-build-isolation -v --no-deps ."
if BACKEND == 'geogram':
    install_cmd = f"GEOGRAM_INSTALL_PREFIX=/usr/local {install_cmd}"
elif BACKEND == 'cgal':
    install_cmd = f"CGALDELAUNAY_ROOT={WORKDIR}/HGP-clusterer/CGALDelaunay CMAKE_PREFIX_PATH={WORKDIR}/HGP-clusterer:{cgal_prefix} {install_cmd}"

print(f"Exécution : {install_cmd}")
!{install_cmd}

os.environ["CGALDELAUNAY_ROOT"] = f"{WORKDIR}/HGP-clusterer/CGALDelaunay"

try:
    import hgp_reducer
    print("\u2705 HGP-Reducer installé.")
except ImportError as e:
    print(f"\u274c Erreur import HGP: {e}")

In [ ]:
# @title 1.2 Ajout du chemin vers HGP-Reducer
import sys
import os
if '/content/HGP-clusterer/src' not in sys.path:
    sys.path.append('/content/HGP-clusterer/src')

In [ ]:
# @title 2. Importation et Génération des Benchmarks 3D
import urllib.request
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll, make_s_curve

# --- Dictionnaire pour stocker les jeux de données : "Nom": (Données_3D, Couleurs)
datasets = {}
n_samples = 5000 # @param {type:"integer"}

# ==========================================
# 1. LE MAMMOUTH (Structure Globale)
# ==========================================
url = "https://raw.githubusercontent.com/PAIR-code/understanding-umap/master/raw_data/mammoth_3d.json"
try:
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response:
        X_mammoth = np.array(json.loads(response.read().decode('utf-8')))

    if len(X_mammoth) > n_samples:
        np.random.seed(42)
        X_mammoth = X_mammoth[np.random.choice(X_mammoth.shape[0], n_samples, replace=False)]

    datasets['1. Mammouth'] = (X_mammoth, X_mammoth[:, 1])
except Exception as e:
    print(f"Erreur téléchargement Mammouth : {e}")

# ==========================================
# 2 & 3. SWISS ROLL & S-CURVE (Dépliage)
# ==========================================
X_swiss, c_swiss = make_swiss_roll(n_samples=n_samples, noise=0.05, random_state=42)
datasets['2. Swiss Roll'] = (X_swiss, c_swiss)

X_scurve, c_scurve = make_s_curve(n_samples=n_samples, noise=0.05, random_state=42)
datasets['3. S-Curve'] = (X_scurve, c_scurve)

# ==========================================
# 4. SPHÈRE TRONQUÉE (Topologie)
# ==========================================
u = np.random.uniform(0, 2 * np.pi, n_samples)
v = np.random.uniform(0.4, np.pi - 0.4, n_samples)
x_sph, y_sph, z_sph = np.sin(v) * np.cos(u), np.sin(v) * np.sin(u), np.cos(v)
datasets['4. Sphère Tronquée'] = (np.vstack((x_sph, y_sph, z_sph)).T, z_sph)

# ==========================================
# 5. NOEUD DE TRÈFLE (Démêlage)
# ==========================================
t = np.random.uniform(0, 2 * np.pi, n_samples)
x_knot = np.sin(t) + 2 * np.sin(2 * t)
y_knot = np.cos(t) - 2 * np.cos(2 * t)
z_knot = -np.sin(3 * t)
X_knot = np.vstack((x_knot, y_knot, z_knot)).T + np.random.normal(0, 0.1, (n_samples, 3))
datasets['5. Nœud de Trèfle'] = (X_knot, t)

# ==========================================
# 6. TORE / DONUT (Déchirure)
# ==========================================
u_tor, v_tor = np.random.uniform(0, 2 * np.pi, n_samples), np.random.uniform(0, 2 * np.pi, n_samples)
R, r = 2.0, 1.0
x_tor = (R + r * np.cos(v_tor)) * np.cos(u_tor)
y_tor = (R + r * np.cos(v_tor)) * np.sin(u_tor)
z_tor = r * np.sin(v_tor)
datasets['6. Tore'] = (np.vstack((x_tor, y_tor, z_tor)).T, u_tor)

# --- Affichage
fig = plt.figure(figsize=(18, 10))
fig.suptitle("La Batterie de Crash-Tests 3D (Vérité Terrain)", fontsize=18, fontweight='bold')

for i, (name, (X, color)) in enumerate(sorted(datasets.items())):
    ax = fig.add_subplot(2, 3, i+1, projection='3d')
    ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=color, cmap='Spectral', s=2, alpha=0.8)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([]); ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# @title 3. Comparaison (PCA, t-SNE, UMAP, HGP-Reducer, HGP-UMAP)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import umap
import numpy as np
from hgp_reducer import HGPReducer
import time

methods = {
    "PCA": PCA(n_components=2, random_state=42),
    "t-SNE": TSNE(n_components=2, init='pca', learning_rate='auto', n_jobs=-1, random_state=42),
    "UMAP": umap.UMAP(n_components=2, n_jobs=-1, random_state=42),
    "HGP-Reducer": HGPReducer(d=2, K=2, expZ=1.0, laplacian_type='normalized', backend=BACKEND, verbose=True),
    "HGP-UMAP": umap.UMAP(n_components=2, n_jobs=-1, random_state=42) # Initialisé dynamiquement
}

n_datasets = len(datasets)
n_methods = len(methods)

fig, axes = plt.subplots(n_methods, n_datasets, figsize=(4 * n_datasets, 4 * n_methods))
fig.suptitle("Comparaison de la Réduction de Dimension", fontsize=24, fontweight='bold')

for col, (name, (X, color)) in enumerate(sorted(datasets.items())):
    axes[0, col].set_title(name, fontsize=16, fontweight='bold')
    hgp_embedding = None

    for row, (method_name, reducer) in enumerate(methods.items()):
        if col == 0:
            axes[row, col].set_ylabel(method_name, fontsize=16, fontweight='bold')

        print(f"Exécution de {method_name} sur {name}...")
        start_time = time.time()
        try:
            if method_name == "HGP-UMAP" and hgp_embedding is not None:
                # Warm-start UMAP with HGP-Reducer's output for speed and better global topology
                init_emb = (hgp_embedding - np.mean(hgp_embedding, axis=0)) / (np.std(hgp_embedding, axis=0) + 1e-8)
                init_emb = init_emb * 10.0
                reducer = umap.UMAP(n_components=2, init=init_emb, n_jobs=-1, random_state=42)
            elif method_name == "HGP-UMAP" and hgp_embedding is None:
                print("Avertissement: HGP-Reducer n'a pas produit d'embedding. HGP-UMAP sera un UMAP standard.")
            
            X_reduced = reducer.fit_transform(X)
            
            if method_name == "HGP-Reducer":
                hgp_embedding = X_reduced.copy()

            end_time = time.time()

            ax = axes[row, col]
            ax.scatter(X_reduced[:, 0], X_reduced[:, 1], c=color, cmap='Spectral', s=2, alpha=0.8)
            ax.set_xticks([]); ax.set_yticks([]); ax.axis('off')

            # Ajouter le temps d'exécution
            ax.text(0.05, 0.05, f"{end_time - start_time:.2f}s", transform=ax.transAxes,
                    fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8))
        except Exception as e:
            print(f"Erreur avec {method_name} sur {name}: {e}")
            axes[row, col].text(0.5, 0.5, "Erreur", ha="center", va="center", color="red")
            axes[row, col].axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()
